# Modélisation

On choisit la cible, on prépare X et y, et on entraîne deux modèles simples
à comparer. L'évaluation poussée (métriques détaillées, interprétation,
ajustement, sauvegarde du modèle) c'est pour le Jour 6 : ici on reste sur
une première version qui tourne, avec juste assez de métriques pour savoir
si on part dans la bonne direction.

## Choix de la cible

On a vu que `estimated_owners_avg` est presque redondant avec
`total_reviews` et `Peak CCU` (fuite de données si on les utilise comme
variables explicatives), alors que `positive_ratio` n'a pas ce problème.
On part donc sur une classification : un jeu est-il "bien noté" ou pas,
à partir d'un seuil sur `positive_ratio`.

Deux précautions :
- On ne garde que les jeux avec au moins 10 avis (`total_reviews >= 10`),
  sinon le ratio est basé sur trop peu d'avis pour être fiable (1 avis sur 1
  donnerait 100%, ce qui est du bruit, pas un vrai signal de qualité)
- On fixe le seuil à 0.8 (proche du seuil "Very Positive" utilisé par Steam
  lui-même), ce qui donne des classes à peu près équilibrées (~53%/47%),
  plus simple à évaluer qu'un déséquilibre fort

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('../data/processed/games_clean.csv', parse_dates=['Release date'])
df['is_indie'] = df['Genres'].fillna('').str.contains('Indie').astype(int)

sub = df[df['total_reviews'] >= 10].copy()
sub['bien_note'] = (sub['positive_ratio'] >= 0.8).astype(int)

print(f"Jeux conservés (>= 10 avis) : {len(sub)} sur {len(df)}")
print(sub['bien_note'].value_counts(normalize=True).rename('proportion'))

Jeux conservés (>= 10 avis) : 56662 sur 125854
bien_note
1    0.534979
0    0.465021
Name: proportion, dtype: float64


## Préparation des features

`main_genre` a 28 valeurs différentes mais très inégalement réparties
(Action, Adventure et Casual concentrent l'essentiel des jeux). On regroupe
les 10 genres principaux et on met le reste dans "Other", pour éviter de
créer des dizaines de colonnes one-hot qui ne contiennent presque aucun jeu.

In [2]:
top_genres = sub['main_genre'].value_counts().head(10).index
sub['main_genre_group'] = sub['main_genre'].where(sub['main_genre'].isin(top_genres), 'Other')

print(sub['main_genre_group'].value_counts())

main_genre_group
Action          23671
Adventure       12741
Casual           9362
Indie            5531
Simulation       1326
Other            1159
RPG               967
Strategy          866
Free To Play      483
Racing            320
Violent           236
Name: count, dtype: int64


In [3]:
features_numeriques = [
    'Price', 'Required age', 'DLC count', 'nb_genres',
    'nb_categories', 'nb_tags', 'release_year', 'is_indie'
]
features_categorielles = ['main_genre_group']

X = sub[features_numeriques + features_categorielles]
y = sub['bien_note']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")

Train : 45329 lignes | Test : 11333 lignes


## Pipeline de prétraitement

On utilise un `ColumnTransformer` plutôt qu'un simple `pd.get_dummies` :
ça évite que les colonnes one-hot du train et du test ne correspondent pas
si une catégorie est absente d'un des deux ensembles, et ça garde la
normalisation (`StandardScaler`) et l'encodage dans le même objet réutilisable
pour la suite (Jour 6, application).

In [4]:
pretraitement = ColumnTransformer([
    ('num', StandardScaler(), features_numeriques),
    ('cat', OneHotEncoder(handle_unknown='ignore'), features_categorielles)
])

## Modèle 1 — Régression logistique

Un modèle simple et rapide, qui sert de référence (baseline) à battre.

In [5]:
pipe_logreg = Pipeline([
    ('prep', pretraitement),
    ('model', LogisticRegression(max_iter=1000))
])

pipe_logreg.fit(X_train, y_train)
pred_logreg = pipe_logreg.predict(X_test)

print("Accuracy :", accuracy_score(y_test, pred_logreg))
print(classification_report(y_test, pred_logreg))

Accuracy : 0.6272831553869231
              precision    recall  f1-score   support

           0       0.62      0.51      0.56      5270
           1       0.63      0.73      0.68      6063

    accuracy                           0.63     11333
   macro avg       0.63      0.62      0.62     11333
weighted avg       0.63      0.63      0.62     11333



## Modèle 2 — Random Forest

Un modèle qui peut capter des interactions plus complexes entre variables
(par exemple "jeu indé ET genre Strategy" plutôt qu'un effet purement additif).

In [6]:
pipe_rf = Pipeline([
    ('prep', pretraitement),
    ('model', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
])

pipe_rf.fit(X_train, y_train)
pred_rf = pipe_rf.predict(X_test)

print("Accuracy :", accuracy_score(y_test, pred_rf))
print(classification_report(y_test, pred_rf))

Accuracy : 0.6318715256331069
              precision    recall  f1-score   support

           0       0.61      0.58      0.59      5270
           1       0.65      0.68      0.66      6063

    accuracy                           0.63     11333
   macro avg       0.63      0.63      0.63     11333
weighted avg       0.63      0.63      0.63     11333



## Premier bilan

Les deux modèles tournent autour de 62-63% d'accuracy, contre 53,5% si on
prédisait toujours la classe majoritaire : il y a bien un signal, mais
modeste. C'est cohérent avec le notebook des corrélations, où aucune variable n'était
fortement corrélée à `positive_ratio`. Random Forest fait un peu mieux et
surtout équilibre mieux les deux classes (meilleur recall sur la classe 0)
que la régression logistique.

À ce stade il ne faut pas chercher à "gonfler" artificiellement
l'accuracy : le message honnête pour l'oral, c'est que les métadonnées
disponibles (prix, genre, nombre de tags...) donnent une indication faible
mais réelle de la réception critique d'un jeu, l'essentiel restant
probablement la qualité du jeu lui-même, que ces données ne capturent pas.